## File paths and train,validation split

In [1]:
import pandas as pd
import tqdm
import numpy as np
import pyspark
import dxpy
import dxdata
import itertools

labels_file = 'labels.csv'
ukb_field_to_icd10_map_file = 'icd10_codes_mod.tsv'

train_proportion = 0.8 # proportion of full data set to use for training (the rest will be used for validation)
output_dir =  # Delphi repository and the data will be put there

## Read icd10 mapping file and defined index label link

In [2]:
# mapping from field name to Delphi token vocabulary index
df_labels = pd.read_csv(labels_file, header=None, sep='\t')[0].rename(lambda x: x - 1)
df_labels = df_labels.str.split(' ').str[0]
label_dict = {v:k for k, v in df_labels.items()}

# UKB field mapping for ICD token, except cancers
df_mapping = pd.read_csv(ukb_field_to_icd10_map_file, sep='\t', header=None, names=['field', 'field_name'])
df_mapping['field'] = 'p' + df_mapping['field'].str.split('.').str[1]
df_mapping['field_name'] = df_mapping['field_name'].str.split().str[4]
icd_codes_dict = df_mapping.set_index('field').to_dict()['field_name']
icd_codes_dict['p40000_i0'] = 'Death' # technically not ICD, but we add death, too

# cancer fields
cancer_data_dict = {}
for j in range(17):
    cancer_data_dict['p40005_i'+str(j)] = "cancer_date_"+str(j)
    cancer_data_dict['p40006_i'+str(j)] = "cancer_type_"+str(j)

# demographics and such 
other_data_dict = {}
other_data_dict['p31'] = "sex"
other_data_dict['p34'] = "YEAR"
other_data_dict['p52'] = "MONTH"

other_data_dict['p53_i0'] = "assessment_date"
other_data_dict['p21001_i0'] = "BMI"
other_data_dict['p1239_i0'] = "smoking"
other_data_dict['p1558_i0'] = "alcohol"

## Retrieve ukb fields from SQL database

In [3]:
def get_database_id():
    dispensed_dataset = dxpy.find_one_data_object(
        typename='Dataset', 
        name='app*.dataset', 
        folder='/', 
        name_mode='glob')
    dispensed_dataset_id = dispensed_dataset['id']
    return(dispensed_dataset_id)

sc = pyspark.SparkContext()
spark = pyspark.sql.SparkSession(sc)

dispensed_dataset_id = get_database_id()

dataset = dxdata.load_dataset(id=dispensed_dataset_id)
participant = dataset['participant']
                     
spark.conf.set("spark.sql.execution.arrow.pyspark.enabled", "true")

#### ICD tokens and death 

In [4]:
fields_chunk_size = 128
chunks = []

for fields_to_get_chunk in itertools.batched(icd_codes_dict.keys(), fields_chunk_size):
    
    data_chunk = participant.retrieve_fields(names=list(fields_to_get_chunk) + ["eid"], 
                                 coding_values = "raw",
                                 engine=dxdata.connect(dialect="hive+pyspark"))
    
    data_chunk = data_chunk.toPandas().set_index("eid")
    data_chunk_long = (data_chunk.stack(future_stack=True)
                                 .dropna()
                                 .rename("date")
                                 .reset_index()
                                 .rename(columns={"level_1": "field_id"}))

    chunks.append(data_chunk_long)
    
data_icd_long = pd.concat(chunks, axis=0).set_index('eid')
data_icd_long['token_name'] = data_icd_long['field_id'].map(icd_codes_dict)

#### Cancer data 

In [5]:
data_cancer = participant.retrieve_fields(names=list(cancer_data_dict.keys()) + ["eid"], 
                             coding_values = "raw",
                             engine=dxdata.connect(dialect="hive+pyspark"))

data_cancer = data_cancer.toPandas().set_index("eid")

In [6]:
date_cols = [f"p40005_i{j}" for j in range(17)]
type_cols = [f"p40006_i{j}" for j in range(17)]

n = len(date_cols)

dates = data_cancer[date_cols].stack().rename("date")
types = data_cancer[type_cols].stack().rename("token_name")

dates.index = dates.index.set_levels(range(n), level=1)
types.index = types.index.set_levels(range(n), level=1)

data_cancer_long = pd.concat([dates, types], axis=1).dropna(how="all").reset_index()
data_cancer_long.columns = ["eid", "instance", "date", "token_name"]
data_cancer_long = data_cancer_long.dropna().set_index('eid').drop(columns=['instance'])

data_cancer_long['token_name'] = data_cancer_long['token_name'].str[:3] # use level 3 ICD codes
data_cancer_long['date'] = pd.to_datetime(data_cancer_long['date'])

#### Demographic and lifestyle data 

In [7]:
data_other = participant.retrieve_fields(names=list(other_data_dict.keys()) + ["eid"], 
                             coding_values = "raw",
                             engine=dxdata.connect(dialect="hive+pyspark"))

data_other = data_other.toPandas().set_index("eid").rename(columns=other_data_dict)

In [ ]:
data_other['date_of_birth'] =  pd.to_datetime(data_other[['YEAR', 'MONTH']].assign(DAY=15)) # we don't have the birthday, so we use the average
data_other['date'] = pd.to_datetime(data_other['assessment_date'])

# discretise lifestyle factors into levels
data_other['BMI'] = pd.cut(data_other['BMI'], [float('-inf'), 22, 28, float('inf')], labels=['BMI_low', 'BMI_mid', 'BMI_high'])

data_other.loc[data_other['smoking'] == -3, 'smoking'] = np.nan # -3 stands for "Prefer not to answer"
data_other['smoking'] = pd.cut(data_other['smoking'], [-1, 0, 1, float('inf')], labels=['Smoking_low', 'Smoking_high', 'Smoking_mid'])

data_other.loc[data_other['alcohol'] == -3, 'alcohol'] = np.nan
data_other['alcohol'] = pd.cut(data_other['alcohol'], [float('-inf'), 1, 3, float('inf')], labels=['Alcohol_high', 'Alcohol_mid', 'Alcohol_low'])

data_other['sex'] = data_other['sex'].map({0: 'Female', 1: 'Male'})

In [9]:
data_lifestyle_long = data_other.reset_index()[['BMI', 'smoking', 'alcohol', 'date', 'sex', 'eid']].melt(id_vars=['date', 'eid']).set_index('eid')
data_lifestyle_long = data_lifestyle_long.rename(columns={'value': 'token_name'})

### Merge all data types together

In [ ]:
columns_to_agg = ['date', 'token_name']

data_all = pd.concat([data_icd_long[columns_to_agg],
           data_lifestyle_long[columns_to_agg],
           data_cancer_long[columns_to_agg]])

data_all['date'] = pd.to_datetime(data_all['date'])

# Remove impossible dates (they usually are defined as special values, but we don't want them)
min_year = 1930
max_year = 2025

data_all = data_all[
    (data_all['date'].dt.year >= min_year) &
    (data_all['date'].dt.year <= max_year)
]

data_all = data_all.merge(data_other[['date_of_birth']], on='eid')

In [ ]:
data_all['age'] = (data_all['date'] - data_all['date_of_birth']).dt.days
data_all = data_all[data_all['age'] >= 0]

# remove all records after death (likely retrospective data)
death_age = data_all.loc[data_all['token_name'] == 'Death', ['age']].rename(columns={'age': 'death_age'})
data_all = data_all.merge(death_age, left_index=True, right_index=True, how='left')
data_all = data_all[(data_all['death_age'].isna()) | (data_all['age'] <= data_all['death_age'])]
data_all = data_all.drop(columns=['death_age'])

data_all.loc[data_all['token_name'].isin(['Male', 'Female']), 'age'] = 0
data_all['token_id'] = data_all['token_name'].map(label_dict)
data_all = data_all.dropna()

## Reformat, split train and val and output to delphi format

In [13]:
data_all = data_all.reset_index()[['eid', 'age', 'token_id']]
data_all = data_all.drop_duplicates(['eid','token_id'])
data_all = data_all.sort_values(['eid', 'age', 'token_id'])

eids = data_all['eid'].unique()
last_train_eid = eids[int(len(eids) * train_proportion)]

data = data_all.values.astype(np.uint32)
train_val_split = (data[:,0] <= int(last_train_eid))
data[train_val_split].tofile('train.bin')
data[~train_val_split].tofile('val.bin')

In [ ]:
!dx upload -r /opt/notebooks/Delphi --destination {output_dir}